In [188]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import gc,sys
import time
from sklearn.preprocessing import normalize
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error

In [112]:
def download(url,filename):
    response=requests.get(url)
    if response.status_code==200:
        with open(filename,'wb') as f:
            f.write(response.content)
    else:
        print(response.status_code)

In [113]:
url='https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-ML0101EN-SkillsNetwork/labs/Module%203/data/yellow_tripdata_2019-06.csv'

In [114]:
filename='yellow_tripdata_2019-06.csv'

In [115]:
download(url,filename)

In [116]:
filename

'yellow_tripdata_2019-06.csv'

In [117]:
raw_data=pd.read_csv(filename)

In [118]:
raw_data.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge
0,1,2019-06-01 00:55:13,2019-06-01 00:56:17,1.0,0.0,1.0,N,145.0,145.0,2.0,3.0,0.5,0.5,0.00,0.0,0.3,4.30,0.0
1,1,2019-06-01 00:06:31,2019-06-01 00:06:52,1.0,0.0,1.0,N,262.0,263.0,2.0,2.5,3.0,0.5,0.00,0.0,0.3,6.30,2.5
2,1,2019-06-01 00:17:05,2019-06-01 00:36:38,1.0,4.4,1.0,N,74.0,7.0,2.0,17.5,0.5,0.5,0.00,0.0,0.3,18.80,0.0
3,1,2019-06-01 00:59:02,2019-06-01 00:59:12,0.0,0.8,1.0,N,145.0,145.0,2.0,2.5,1.0,0.5,0.00,0.0,0.3,4.30,0.0
4,1,2019-06-01 00:03:25,2019-06-01 00:15:42,1.0,1.7,1.0,N,113.0,148.0,1.0,9.5,3.0,0.5,2.65,0.0,0.3,15.95,2.5


In [119]:
print("There are"+str(len(raw_data))+"observations in the datasets")
print("There are"+str(len(raw_data))+"observations in the datasets")

There are3936004observations in the datasets
There are3936004observations in the datasets


In [120]:
raw_data=raw_data[raw_data['tip_amount']>0]
raw_data=raw_data[(raw_data['tip_amount']<=raw_data['fare_amount'])]

raw_data=raw_data[((raw_data['fare_amount']>=2) & (raw_data['fare_amount']<200))]

clean_data=raw_data.drop(['total_amount'],axis=1)


del raw_data

gc.collect()

print("There are"+str(len(clean_data))+"observations in the dataset")
print("There are"+str(len(clean_data.columns))+"variables in the dataset")



There are2712719observations in the dataset
There are17variables in the dataset


In [121]:
clean_data.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,congestion_surcharge
4,1,2019-06-01 00:03:25,2019-06-01 00:15:42,1.0,1.70,1.0,N,113.0,148.0,1.0,9.5,3.0,0.5,2.65,0.0,0.3,2.5
5,1,2019-06-01 00:28:31,2019-06-01 00:39:23,2.0,1.60,1.0,N,79.0,125.0,1.0,9.5,3.0,0.5,1.00,0.0,0.3,2.5
7,1,2019-06-01 00:54:49,2019-06-01 01:02:57,2.0,1.20,1.0,N,79.0,249.0,1.0,7.5,3.0,0.5,1.00,0.0,0.3,2.5
9,1,2019-06-01 00:29:12,2019-06-01 01:03:13,1.0,8.60,1.0,N,186.0,243.0,1.0,31.5,3.0,0.5,7.05,0.0,0.3,2.5
10,2,2019-06-01 00:01:48,2019-06-01 00:16:06,1.0,1.74,1.0,N,107.0,148.0,1.0,11.0,0.5,0.5,2.96,0.0,0.3,2.5


In [122]:
clean_data['tpep_dropoff_datetime']=pd.to_datetime(clean_data['tpep_dropoff_datetime'])
clean_data['tpep_pickup_datetime']=pd.to_datetime(clean_data['tpep_pickup_datetime'])


clean_data['pickup_hour']=clean_data['tpep_pickup_datetime'].dt.hour
clean_data['dropoff_hour']=clean_data['tpep_dropoff_datetime'].dt.hour

clean_data['pickup_day']=clean_data['tpep_pickup_datetime'].dt.weekday
clean_data['dropoff_day']=clean_data['tpep_dropoff_datetime'].dt.weekday

clean_data['trip_time']=(clean_data['tpep_dropoff_datetime']-clean_data['tpep_pickup_datetime']).dt.total_seconds() / 60

clean_data=clean_data.head()

In [123]:
clean_data.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,mta_tax,tip_amount,tolls_amount,improvement_surcharge,congestion_surcharge,pickup_hour,dropoff_hour,pickup_day,dropoff_day,trip_time
4,1,2019-06-01 00:03:25,2019-06-01 00:15:42,1.0,1.70,1.0,N,113.0,148.0,1.0,...,0.5,2.65,0.0,0.3,2.5,0,0,5,5,12.283333
5,1,2019-06-01 00:28:31,2019-06-01 00:39:23,2.0,1.60,1.0,N,79.0,125.0,1.0,...,0.5,1.00,0.0,0.3,2.5,0,0,5,5,10.866667
7,1,2019-06-01 00:54:49,2019-06-01 01:02:57,2.0,1.20,1.0,N,79.0,249.0,1.0,...,0.5,1.00,0.0,0.3,2.5,0,1,5,5,8.133333
9,1,2019-06-01 00:29:12,2019-06-01 01:03:13,1.0,8.60,1.0,N,186.0,243.0,1.0,...,0.5,7.05,0.0,0.3,2.5,0,1,5,5,34.016667
10,2,2019-06-01 00:01:48,2019-06-01 00:16:06,1.0,1.74,1.0,N,107.0,148.0,1.0,...,0.5,2.96,0.0,0.3,2.5,0,0,5,5,14.300000


In [124]:
clean_data=clean_data.drop(['tpep_pickup_datetime','tpep_dropoff_datetime'],axis=1)

In [125]:
clean_data.head()

,VendorID,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,congestion_surcharge,pickup_hour,dropoff_hour,pickup_day,dropoff_day,trip_time
4,1,1.0,1.70,1.0,N,113.0,148.0,1.0,9.5,3.0,0.5,2.65,0.0,0.3,2.5,0,0,5,5,12.283333
5,1,2.0,1.60,1.0,N,79.0,125.0,1.0,9.5,3.0,0.5,1.00,0.0,0.3,2.5,0,0,5,5,10.866667
7,1,2.0,1.20,1.0,N,79.0,249.0,1.0,7.5,3.0,0.5,1.00,0.0,0.3,2.5,0,1,5,5,8.133333
9,1,1.0,8.60,1.0,N,186.0,243.0,1.0,31.5,3.0,0.5,7.05,0.0,0.3,2.5,0,1,5,5,34.016667
10,2,1.0,1.74,1.0,N,107.0,148.0,1.0,11.0,0.5,0.5,2.96,0.0,0.3,2.5,0,0,5,5,14.300000


In [156]:
get_dummy_col = ["VendorID","RatecodeID","store_and_fwd_flag","PULocationID", "DOLocationID","payment_type", "pickup_hour", "dropoff_hour", "pickup_day", "dropoff_day"]

proc_data=pd.get_dummies(clean_data,columns=get_dummy_col)

del clean_data
gc.collect()

16197

In [162]:
y=proc_data[['tip_amount']].values.astype('float32')

In [164]:
proc_data=proc_data.drop(['tip_amount'],axis=1)

In [166]:
proc_data.head()

,passenger_count,trip_distance,fare_amount,extra,mta_tax,tolls_amount,improvement_surcharge,congestion_surcharge,trip_time,VendorID_1,...,DOLocationID_125.0,DOLocationID_148.0,DOLocationID_243.0,DOLocationID_249.0,payment_type_1.0,pickup_hour_0,dropoff_hour_0,dropoff_hour_1,pickup_day_5,dropoff_day_5
4,1.0,1.70,9.5,3.0,0.5,0.0,0.3,2.5,12.283333,True,...,False,True,False,False,True,True,True,False,True,True
5,2.0,1.60,9.5,3.0,0.5,0.0,0.3,2.5,10.866667,True,...,True,False,False,False,True,True,True,False,True,True
7,2.0,1.20,7.5,3.0,0.5,0.0,0.3,2.5,8.133333,True,...,False,False,False,True,True,True,False,True,True,True
9,1.0,8.60,31.5,3.0,0.5,0.0,0.3,2.5,34.016667,True,...,False,False,True,False,True,True,False,True,True,True
10,1.0,1.74,11.0,0.5,0.5,0.0,0.3,2.5,14.300000,False,...,False,True,False,False,True,True,True,False,True,True


In [168]:
X=proc_data.values
X=normalize(X,axis=1,norm='l1',copy=False)

print('X shape:',X.shape,'y.shape',y.shape)

X shape: (5, 27) y.shape (5, 1)


In [178]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=42)
print(f"X_train.shape:{X_train.shape}\ny_train.shape:{y_train.shape}")
print(f"X_test.shape:{X_test.shape},\ny_test.shape:{y_test.shape}")

X_train.shape:(3, 27)
y_train.shape:(3, 1)
X_test.shape:(2, 27),
y_test.shape:(2, 1)


In [186]:
sklearn_dt=DecisionTreeRegressor(max_depth=8,random_state=35)
t0=time.time()
sklearn_dt.fit(X_train,y_train)
sklearn_time=time.time()-t0
print("[Scikit-Learn] Training time (s):  {0:.5f}".format(sklearn_time))

[Scikit-Learn] Training time (s):  0.00657


In [190]:
sklearn_pred=sklearn_dt.predict(X_test)
sklearn_mse=mean_squared_error(y_test,sklearn_pred)
print('[Scikit-Learn] MSE score : {0:.3f}'.format(sklearn_mse))

[Scikit-Learn] MSE score : 1.409
